## Preparation (3 points total)

[1 points] Define and prepare your class variables. Use proper variable representations (int, float, one-hot, etc.). Use pre-processing methods (as needed). Describe the final dataset that is used for classification/regression (include a description of any newly formed variables you created). Discuss methods of tokenization in your dataset as well as any decisions to force a specific length of sequence.  

[1 points] Choose and explain what metric(s) you will use to evaluate your algorithm’s performance. You should give a detailed argument for why this (these) metric(s) are appropriate on your data. That is, why is the metric appropriate for the task (e.g., in terms of the business case for the task). Please note: rarely is accuracy the best evaluation metric to use. Think deeply about an appropriate measure of performance.

[1 points] Choose the method you will use for dividing your data into training and testing (i.e., are you using Stratified 10-fold cross validation? Shuffle splits? Why?). Explain why your chosen method is appropriate or use more than one method as appropriate. Convince me that your train/test splitting method is a realistic mirroring of how an algorithm would be used in practice. 

## 


# Preparation Work

The following section completes the Preparation portion of Lab 7 by selecting a text dataset, preparing the class variable, tokenizing the text, choosing a fixed sequence length, selecting evaluation metrics, and creating a realistic train/validation/test split.


In [ ]:

import os
import random
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:

os.makedirs("data", exist_ok=True)

url = "https://raw.githubusercontent.com/susanli2016/PyCon-Canada-2019-NLP-Tutorial/master/bbc-text.csv"
file_path = "data/bbc-text.csv"

if not os.path.exists(file_path):
    urllib.request.urlretrieve(url, file_path)

df = pd.read_csv(file_path)

print(df.head())
print("\nDataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:")
print(df.isna().sum())
print("\nClass distribution:")
print(df["category"].value_counts())



## Sample Text Examples

The following examples show representative BBC news articles from different categories. These examples illustrate the structure and variability of the documents used for sequence classification.


In [ ]:

sample_df = df.sample(5, random_state=SEED)[["category", "text"]]

for i, row in sample_df.iterrows():
    print("=" * 80)
    print("Category:", row["category"])
    print()
    print(row["text"][:300] + "...")
    print()


In [ ]:

df["word_count"] = df["text"].str.split().str.len()

print(df["word_count"].describe())

plt.figure(figsize=(8, 5))
plt.hist(df["word_count"], bins=40)
plt.title("Distribution of BBC Article Word Counts")
plt.xlabel("Word Count")
plt.ylabel("Number of Articles")
plt.show()



## Class Variable Preparation

The class variable is the `category` column, which contains five BBC news topic labels: business, entertainment, politics, sport, and tech. Since neural networks require numeric targets, the labels are encoded as integers from 0 to 4. The input variable is the article text.


In [ ]:

label_encoder = LabelEncoder()
df["category_encoded"] = label_encoder.fit_transform(df["category"])

class_names = label_encoder.classes_
num_classes = len(class_names)

print("Class mapping:")
for idx, label in enumerate(class_names):
    print(idx, "=", label)

X_text = df["text"].astype(str).values
y = df["category_encoded"].values

print("\nNumber of classes:", num_classes)



## Tokenization and Sequence Length

The text is tokenized using Keras `Tokenizer`, which converts words into integer indices based on vocabulary frequency. The vocabulary is limited to the 20,000 most common words. A maximum sequence length of 500 tokens is used because the median article length is around 337 words and the mean is around 390 words, while a small number of articles are much longer. Shorter articles are padded and longer articles are truncated.


In [ ]:

MAX_WORDS = 20000
MAX_SEQUENCE_LENGTH = 500

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_text)

sequences = tokenizer.texts_to_sequences(X_text)
X = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH, padding="post", truncating="post")

word_index = tokenizer.word_index
vocab_size = min(MAX_WORDS, len(word_index) + 1)

print("Vocabulary size used:", vocab_size)
print("Padded sequence shape:", X.shape)
print("Target shape:", y.shape)



## Train, Validation, and Test Split

A stratified 70/15/15 train-validation-test split is used. Stratification is appropriate because the news categories are not perfectly balanced, and each split should preserve the original class distribution. The validation set is used for tuning, while the test set remains held out for final evaluation.


In [ ]:

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

print("\nTraining class counts:", np.bincount(y_train))
print("Validation class counts:", np.bincount(y_val))
print("Test class counts:", np.bincount(y_test))



## Evaluation Metric

The main evaluation metric will be macro F1-score. Macro F1 is appropriate because this is a multi-class classification task and the classes are somewhat imbalanced. Macro F1 evaluates each class equally, so the model must perform well across all five categories rather than only performing well on the largest categories. Accuracy will also be reported, but it is not the primary metric.


Modeling (6 points total)

[3 points] Investigate at least two different sequential network architectures (e.g., a CNN and a Transformer). Alternatively, you may also choose a recurrent network and Transformer network. Be sure to use an embedding layer (try to use a pre-trained embedding, if possible). Adjust one hyper-parameter of each network to potentially improve generalization performance (train a total of at least four models). Visualize the performance of training and validation sets versus the training iterations, showing that the models converged.

[1 points] Using the best parameters and architecture from the Transformer in the previous step, add a second Multi-headed self attention layer to your network. That is, the input to the second attention layer should be the output sequence of the first attention layer.  Visualize the performance of training and validation sets versus the training iterations, showing that the model converged.. 

[2 points] Use the method of train/test splitting and evaluation criteria that you argued for at the beginning of the lab. Visualize the results of all the models you trained.  Use proper statistical comparison techniques to determine which method(s) is (are) superior.  

## Exceptional Work (1 points total)

You have free reign to provide additional analyses.
One idea (required for 7000 level students to do one of these options):
Use the pre-trained ConceptNet Numberbatch embedding and compare to pre-trained GloVe. Which method is better for your specific application? 